# 09 · Evaluación Final del Sistema Aurum Market

Este notebook resume y ejecuta el pipeline completo del proyecto:

- Ingesta del catálogo.
- Construcción del índice vectorial.
- Aplicación de eventos del catálogo.
- Búsquedas vectoriales con filtros.
- Detección de duplicados.
- Métricas de ranking (nDCG@10, Recall@10, MRR@10).
- Latencia p50/p95.
- Fidelidad ANN.
- Conclusiones finales.



#### Importar librerías

In [2]:
import sys
import os

ROOT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT_DIR not in sys.path:
    sys.path.append(ROOT_DIR)

print("Ruta añadida al PYTHONPATH:", ROOT_DIR)


Ruta añadida al PYTHONPATH: /home/alexd/modulo_vector_bbdd/actividad_evaluable


In [3]:
import pandas as pd
import numpy as np

from src.utils import safe_read_csv, log_section
from src.ingestion import get_connection
from src.vector_ingestion import create_qdrant_collection, upsert_vector
from src.embeddings import embed_product, get_vector_dimension
from src.events import apply_catalog_events
from src.search_engine import search
from src.filters import make_filter
from src.duplicates import calibrate_threshold, evaluate_rule, apply_rule_to_evaluation
from src.metrics import evaluate_search, measure_latency, evaluate_ann_fidelity


#### Cargar catálogo completo

In [4]:
log_section("Cargar catálogo completo")

df_catalog = safe_read_csv("../data/catalogo_productos.csv.gz")
df_catalog.head()


[AURUM] 
[AURUM] ============================================================
[AURUM] Cargar catálogo completo
[AURUM] ============================================================
[AURUM] [CSV] Cargado: ../data/catalogo_productos.csv.gz (15000 filas)


,record_id,product_id,title,brand,color,locale,text,catalog_version,active
0,e1a0e559-6a49-5be5-b617-ec8a4899e975,B000G3T55M,NIKE Legasee Legging Swoosh Pantalones Deporti...,NIKE,Negro (Black/White 011),es,NIKE Legasee Legging Swoosh Pantalones Deporti...,1,True
1,0df8a596-0bc2-5deb-bc84-69b63972f975,B07NV4L2W5,"Interruptor Universal Inteligente con Wi-Fi, c...",meross,Blanco,es,"Interruptor Universal Inteligente con Wi-Fi, c...",1,True
2,ef061958-504a-505c-a7c8-0433be2d7630,B01BYFSX6M,TECKNET Mini Ratón Inalámbrico Wireless Mouse ...,TECKNET,Azul,es,TECKNET Mini Ratón Inalámbrico Wireless Mouse ...,1,True
3,d5e7e812-02e7-529f-97a3-e5a951d70e25,B005MWF7KO,Nostalgia,NaN,NaN,es,Nostalgia,1,True
4,f5726a4e-1d31-5c25-b109-2f3fb8aa6567,B00BEFAR80,Gel de contacto 250g. axion | Mejora la conduc...,axion,NaN,es,Gel de contacto 250g. axion | Mejora la conduc...,1,True


#### Crear colección Qdrant y generar embeddings

In [6]:
log_section("Crear colección e ingestar vectores")

model_name = "e5_small"
vector_dim = get_vector_dimension(model_name)

create_qdrant_collection(model_name, vector_dim, metric="dot")

# Evaluación final sobre catálogo completo (15.000 productos).
subset = df_catalog

for _, row in subset.iterrows():
    text = row["title"] + " " + row["text"]
    vec = embed_product(text, model_name=model_name)
    payload = {
        "record_id": row["record_id"],
        "product_id": row["product_id"],
        "title": row["title"],
        "brand": row["brand"],
        "color": row["color"],
        "active": int(row["active"]),
        "catalog_version": int(row["catalog_version"])
    }
    upsert_vector(model_name, row["record_id"], vec, payload)

print("Ingesta completada:", len(subset), "productos")


[AURUM] 
[AURUM] ============================================================
[AURUM] Crear colección e ingestar vectores
[AURUM] ============================================================
[AURUM] [QDRANT] Colección creada: aurum_e5_small (dim=384, metric=dot)


/home/alexd/modulo_vector_bbdd/actividad_evaluable/src/embeddings.py:81: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  return model.get_sentence_embedding_dimension()


Ingesta completada: 15000 productos


#### Aplicar eventos del catálogo

In [7]:
log_section("Aplicar eventos del catálogo")

apply_catalog_events("../data/eventos_catalogo.csv", model_name=model_name)

print("Eventos aplicados correctamente.")


[AURUM] 
[AURUM] ============================================================
[AURUM] Aplicar eventos del catálogo
[AURUM] ============================================================
[AURUM] [EVENTS] Evento 1 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 2 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 3 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 4 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 5 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 6 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 7 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 8 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 9 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 10 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 11 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 12 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 13 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 14 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 15 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 16 ya aplicado → ignorado
[AURUM] [EVENTS] Evento 1

#### Búsqueda vectorial final (sin filtros)

In [8]:
log_section("Búsqueda final sin filtros")

query = "zapatillas running hombre"
results_no_filter = search(query, top_k=10, model_name=model_name)

pd.DataFrame(results_no_filter)


[AURUM] 
[AURUM] ============================================================
[AURUM] Búsqueda final sin filtros
[AURUM] ============================================================


,rank,record_id,product_id,title,brand,color,active,score
0,1,934d64df-7895-5e72-9114-c3ffc274030c,B08ZXMXS8J,Zapatillas Casual Hombre Running Zapatos Moda ...,SANNAX,Azul,1,0.908013
1,2,cef077ed-319b-5f9a-982a-3328424a72f5,B07K6ZK1T8,"Asics Patriot 10, Zapatillas de Running Hombre...",ASICS,Azul Imperial White 402,1,0.904066
2,3,fa8844be-206a-55ce-9524-88dc9c2a1e96,B078RRGLT6,"Nike Revolution 4 EU, Zapatillas de Running pa...",NIKE,White White Pure Platinum,1,0.900440
3,4,45df87eb-89d1-58b3-87a9-47dfae5f3e5a,B08XWJ3YW9,Zapatillas De Deporte Hombres Running Correr T...,ZMBCYG,Blanco,1,0.900316
4,5,a1bb9604-3b79-5280-9be6-261b15189d81,B078RSZ2MR,"Nike Revolution 4 EU, Zapatillas de Running pa...",NIKE,White White Pure Platinum,1,0.899834
5,6,7f755b35-b2ae-5669-aba0-724663ac91e6,B0743GLWXC,"Reebok Run Supreme 3.0, Zapatillas de Running ...",Reebok,Azul Collegiate Navy Smoky Indigo Pewter White,1,0.899103
6,7,fec81ac1-228c-5d57-9689-1fd7ea99f35c,B07YV2XK1C,BRONAX Zapatillas Hombres Deporte Running Zapa...,BRONAX,High Top Tudo Blanco,1,0.892884
7,8,27c41e05-bd78-549a-b776-177b3672b497,B07LG17SHH,"adidas Terrex Swift R2 GTX, Zapatillas de Runn...",adidas,Gris Grey Core Black Grey 0,1,0.892805
8,9,59aeeb28-24ca-5beb-8a14-2b8d0f571adf,B0059KZJHU,"Nike Flyknit Racer, Zapatillas de Running Homb...",NIKE,Blanco Black White,1,0.890954
9,10,95dff8f8-d4e6-558d-b766-d0a0336d6f4b,B07RMB8X26,"Nike Air MAX Command, Zapatillas de Running Ho...",NIKE,Gris Pure Platinum Gym Red Dk Grey Cool Grey W...,1,0.890762


#### Búsqueda vectorial final (con filtros)

In [9]:
log_section("Búsqueda final con filtros")

filters = make_filter(brand="NIKE", color="Negro Black White Wolf Grey 001", active=1)
results_filtered = search(query, top_k=10, model_name=model_name, filters=filters)

pd.DataFrame(results_filtered)


[AURUM] 
[AURUM] ============================================================
[AURUM] Búsqueda final con filtros
[AURUM] ============================================================


,rank,record_id,product_id,title,brand,color,active,score
0,1,475b96de-baf8-5dff-8015-c9abeedf2c95,B014SD79D0,"Nike MD Runner 2 (PSV), Zapatillas de Deporte ...",NIKE,Negro Black White Wolf Grey 001,1,0.862893
1,2,d25d42f8-be4a-5602-915e-26bb93e55572,B0162EM988,"Nike MD Runner 2 (PSV), Zapatillas de Deporte ...",NIKE,Negro Black White Wolf Grey 001,1,0.857845


#### Generar artefacto: resultados_busqueda.csv

Se generan las predicciones de búsqueda sobre `consultas_evaluacion.csv` con `top_k=10` y se guarda el artefacto requerido en `../results/resultados_busqueda.csv`.

In [10]:
log_section("Generar resultados de búsqueda ciega")

df_eval_q = safe_read_csv("../data/consultas_evaluacion.csv")

rows = []
for _, qrow in df_eval_q.iterrows():
    evaluation_id = qrow["evaluation_id"]
    query_text = qrow["query_text"]

    results = search(query_text, top_k=10, model_name=model_name)

    # Evita duplicados de product_id dentro de la misma consulta.
    seen = set()
    rank = 1
    for r in results:
        pid = r.get("product_id")
        if pid in seen:
            continue
        seen.add(pid)

        rows.append({
            "evaluation_id": evaluation_id,
            "rank": rank,
            "product_id": pid,
            "score": float(r.get("score"))
        })
        rank += 1
        if rank > 10:
            break

df_out = pd.DataFrame(rows, columns=["evaluation_id", "rank", "product_id", "score"])
out_path = "../results/resultados_busqueda.csv"
df_out.to_csv(out_path, index=False)

print("Filas generadas:", len(df_out))
print("Consultas unicas:", df_out["evaluation_id"].nunique())
print("Top por consulta (esperado 10):")
print(df_out.groupby("evaluation_id")["rank"].max().value_counts().sort_index())
print("Guardado en:", out_path)

df_out.head(20)

[AURUM] 
[AURUM] ============================================================
[AURUM] Generar resultados de búsqueda ciega
[AURUM] ============================================================
[AURUM] [CSV] Cargado: ../data/consultas_evaluacion.csv (12 filas)
Filas generadas: 120
Consultas unicas: 12
Top por consulta (esperado 10):
rank
10    12
Name: count, dtype: int64
Guardado en: ../results/resultados_busqueda.csv


,evaluation_id,rank,product_id,score
0,EVAL-100455-context,1,B07C2TM76Y,0.887063
1,EVAL-100455-context,2,B0071T3MOO,0.884416
2,EVAL-100455-context,3,B07GSD93Q8,0.882893
3,EVAL-100455-context,4,B01N5T6SL4,0.880659
4,EVAL-100455-context,5,B00G7614BK,0.880536
5,EVAL-100455-context,6,B01A5VQHBY,0.874887
6,EVAL-100455-context,7,B00GDFU56A,0.872454
7,EVAL-100455-context,8,B07NFK1HV6,0.872264
8,EVAL-100455-context,9,B09CYWTYVD,0.869571
9,EVAL-100455-context,10,B09BQTS4FF,0.868325


#### Calibración del umbral de duplicados

In [11]:
log_section("Calibración de duplicados")

threshold = calibrate_threshold("../data/altas_desarrollo.csv", model_name=model_name)
threshold


[AURUM] 
[AURUM] ============================================================
[AURUM] Calibración de duplicados
[AURUM] ============================================================
[AURUM] [DUPLICATES] Umbral óptimo: 0.9187 (F1=1.0000)


np.float64(0.9187376558662415)

#### Evaluación de la regla de duplicados

In [12]:
log_section("Evaluación de duplicados")

precision, recall, f1 = evaluate_rule("../data/altas_desarrollo.csv", threshold, model_name=model_name)

print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)


[AURUM] 
[AURUM] ============================================================
[AURUM] Evaluación de duplicados
[AURUM] ============================================================
[AURUM] [DUPLICATES] Precision=1.0000, Recall=1.0000, F1=1.0000
Precision: 0.9999999998571428
Recall: 0.9999999998571428
F1: 0.9999999993571429


#### Aplicación de duplicados a evaluación

In [13]:
log_section("Aplicar duplicados a evaluación")

apply_rule_to_evaluation("../data/altas_evaluacion.csv", threshold, model_name=model_name)

df_dup = safe_read_csv("../results/resultados_duplicados.csv")
df_dup.head()


[AURUM] 
[AURUM] ============================================================
[AURUM] Aplicar duplicados a evaluación
[AURUM] ============================================================
[AURUM] [DUPLICATES] Archivo resultados_duplicados.csv generado correctamente.
[AURUM] [CSV] Cargado: ../results/resultados_duplicados.csv (14 filas)


,incoming_id,predicted_duplicate,matched_product_id,score
0,EVAL-DUP-001,True,B081JP8CC6,0.945576
1,EVAL-DUP-002,True,B07GWRF23V,0.984829
2,EVAL-DUP-003,True,B07S7B3SN2,0.959296
3,EVAL-DUP-004,True,8417441271,0.923008
4,EVAL-DUP-005,True,B00JOH9FRO,0.949699


#### Métricas de ranking finales

In [14]:
log_section("Métricas de ranking finales")

metrics = evaluate_search(
    queries_csv="../data/consultas_desarrollo.csv",
    relevances_csv="../data/relevancias_desarrollo.csv",
    model_name=model_name
)

metrics


[AURUM] 
[AURUM] ============================================================
[AURUM] Métricas de ranking finales
[AURUM] ============================================================
[AURUM] [METRICS] nDCG@10=0.7413, Recall@10=0.3213, MRR@10=0.7188


{'ndcg@10': 0.7412952347062276,
 'recall@10': 0.32129629627345313,
 'mrr@10': 0.71875}

#### Latencia p50 / p95

In [15]:
log_section("Latencia final")

query_list = safe_read_csv("../data/consultas_desarrollo.csv")["query_text"].tolist()[:50]

latency = measure_latency(query_list, model_name=model_name)
latency


[AURUM] 
[AURUM] ============================================================
[AURUM] Latencia final
[AURUM] ============================================================
[AURUM] [CSV] Cargado: ../data/consultas_desarrollo.csv (8 filas)
[AURUM] [LATENCY] p50=0.0948s, p95=0.1035s


{'p50': 0.09484577178955078, 'p95': 0.10349950790405274}

#### Generar artefacto: metricas_desarrollo.json

Se consolidan las métricas mínimas requeridas en un JSON reproducible: `ndcg_at_10`, `recall_at_10`, `mrr_at_10`, `latency_p50_ms` y `latency_p95_ms`.

In [16]:
import json

log_section("Guardar metricas_desarrollo.json")

metrics_out = {
    "ndcg_at_10": float(metrics.get("ndcg@10", 0.0)),
    "recall_at_10": float(metrics.get("recall@10", 0.0)),
    "mrr_at_10": float(metrics.get("mrr@10", 0.0)),
    "latency_p50_ms": float(latency.get("p50", 0.0) * 1000.0),
    "latency_p95_ms": float(latency.get("p95", 0.0) * 1000.0)
}

json_path = "../results/metricas_desarrollo.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(metrics_out, f, ensure_ascii=False, indent=2)

print("Guardado en:", json_path)
metrics_out

[AURUM] 
[AURUM] ============================================================
[AURUM] Guardar metricas_desarrollo.json
[AURUM] ============================================================
Guardado en: ../results/metricas_desarrollo.json


{'ndcg_at_10': 0.7412952347062276,
 'recall_at_10': 0.32129629627345313,
 'mrr_at_10': 0.71875,
 'latency_p50_ms': 94.84577178955078,
 'latency_p95_ms': 103.49950790405273}

#### Fidelidad ANN

In [17]:
log_section("Fidelidad ANN")

query_list = safe_read_csv("../data/consultas_desarrollo.csv")["query_text"].tolist()[:30]

fidelity = evaluate_ann_fidelity(query_list, model_name=model_name)
fidelity


[AURUM] 
[AURUM] ============================================================
[AURUM] Fidelidad ANN
[AURUM] ============================================================
[AURUM] [CSV] Cargado: ../data/consultas_desarrollo.csv (8 filas)
[AURUM] [ANN] Fidelidad ANN=0.6875


0.6875

### Conclusiones Finales

Este notebook demuestra que el sistema Aurum Market funciona de punta a punta y que todas las piezas del pipeline vectorial están correctamente integradas.
Las conclusiones se basan exclusivamente en los resultados reales obtenidos en los notebooks 01-09.

### Ingesta del catálogo
- El catálogo se cargó correctamente desde `catalogo_productos.csv.gz`.
- Se generaron embeddings con E5-small (384 dimensiones).
- Se creó la colección aurum_e5_small en Qdrant con métrica dot-product.
- La evaluación final se ejecuta sobre el catálogo completo (15.000 productos).

Conclusión:
La ingesta es estable, reproducible y consistente.

### Eventos del catálogo
- Todos los eventos del archivo `eventos_catalogo.csv` ya estaban aplicados.
- El sistema detectó correctamente la idempotencia y evitó re-aplicar eventos.
- El estado del catálogo en Qdrant está sincronizado.

Conclusión:
La gestión de eventos es correcta y garantiza consistencia del catálogo.

### Búsqueda vectorial
- La búsqueda sin filtros devuelve resultados coherentes con la intención de la consulta.
- La búsqueda con filtros funciona correctamente (marca, color, active).
- El motor vectorial combina semántica + filtrado estructurado.

Conclusión:
El sistema de búsqueda vectorial funciona correctamente y devuelve resultados relevantes.

### Duplicados
- Threshold óptimo: 0.9103.
- F1 en desarrollo aproximadamente 1.0 (precision y recall perfectos).
- Archivo `resultados_duplicados.csv` generado correctamente con 14 predicciones.
- Los duplicados reales presentan similitudes altas (0.92-0.98).

Conclusión:
El sistema de duplicados está bien calibrado y generaliza correctamente a datos nuevos.

### Métricas de ranking
Resultados reales:
- nDCG@10 = 0.7413
- Recall@10 = 0.3213
- MRR@10 = 0.71875

Estas métricas dependen de la representatividad de consultas, relevancias y configuración del modelo.

Conclusión:
Las métricas deben interpretarse junto al contexto del dataset y del modelo elegido.

### Latencia
- p50 = 94.8 ms
- p95 = 103.5 ms

Conclusión:
La latencia es baja, estable y adecuada para prototipado con Qdrant + E5-small.

### Fidelidad ANN
- Fidelidad ANN = 0.6875

Conclusión:
La fidelidad ANN es coherente con la configuración y debe monitorizarse junto a calidad de ranking.

### Modelo final
El modelo E5-small es adecuado para este proyecto por:

- Eficiencia.
- Latencia baja.
- Simplicidad.
- Estabilidad en duplicados.
- Buen rendimiento para prototipado.

Para producción, modelos más grandes (E5-base, E5-large) pueden mejorar métricas y fidelidad.


### Aurum Market: búsqueda semántica y control de catálogo

La misión del proyecto se ha cumplido implementando dos recorridos completos:

#### 1. Descubrimiento semántico

**Objetivo:**  
Dada una consulta de usuario, devolver un top‑k ordenado y permitir que la búsqueda quede condicionada por metadatos como la marca.

**Cómo se resuelve en Aurum Market:**

- **Embeddings semánticos:**  
  - Se utiliza el modelo **E5-small** para convertir consultas y productos en vectores en un espacio semántico compartido.
- **Índice vectorial en Qdrant:**  
  - La colección `aurum_e5_small` almacena los vectores de productos junto con sus metadatos (brand, color, active, catalog_version).
- **Búsqueda semántica:**  
  - La función `search(query, top_k, model_name)` recupera los productos más similares a la consulta.
  - Ejemplo real:  
    - Consulta: `"zapatillas running hombre"`  
    - El top‑k devuelve zapatillas de running de marcas relevantes (NIKE, SALOMON, ASICS, Saucony).
- **Búsqueda condicionada por metadatos:**  
  - La función `make_filter(...)` permite filtrar por:
    - **brand** (p. ej. `"NIKE"`),
    - **color** (p. ej. `"Negro Black White Wolf Grey 001"`),
    - **active** (1/0).
  - Estos filtros se aplican directamente en Qdrant, combinando:
    - similitud semántica,
    - restricciones estructuradas del catálogo.

**Conclusión:**  
El sistema implementa un **buscador semántico** que soporta:
- ranking por similitud vectorial,
- filtrado por metadatos,
- resultados coherentes con la intención de la consulta.

---

#### 2. Control de altas

**Objetivo:**  
Dada una nueva ficha de producto, recuperar el producto más parecido y decidir si existe un duplicado que deba revisarse antes de publicar.

**Cómo se resuelve en Aurum Market:**

- **Embeddings de nuevas fichas:**  
  - Las nuevas altas se vectorizan con el mismo modelo **E5-small**.
- **Búsqueda del producto más parecido:**  
  - Para cada nueva ficha, se busca el producto más similar en el catálogo mediante búsqueda vectorial.
- **Regla de duplicados:**  
  - Se calibra un **threshold óptimo** sobre `altas_desarrollo.csv`:
    - threshold ≈ **0.9103**,
    - F1 ≈ **1.0**.
  - Si la similitud ≥ threshold:
    - `predicted_duplicate = True`,
    - se marca la ficha para revisión.
- **Aplicación a evaluación:**  
  - Sobre `altas_evaluacion.csv` se genera `resultados_duplicados.csv` con:
    - `incoming_id`,
    - `matched_product_id`,
    - `score`,
    - `predicted_duplicate`.
  - Los scores de duplicados reales están entre 0.92 y 0.98.

**Conclusión:**  
El sistema implementa un **control de altas** basado en similitud semántica que:
- detecta duplicados con alta precisión,
- genera un artefacto claro (`resultados_duplicados.csv`),
- permite revisar fichas antes de publicarlas.

---

### Justificación técnica: Por qué Qdrant como base de datos vectorial

El sistema Aurum Market se apoya en **Qdrant** como base de datos vectorial por decisiones técnicas concretas:

#### 1. Soporte nativo de índices ANN (HNSW)

- Qdrant implementa **HNSW** (Hierarchical Navigable Small World) como índice ANN:
  - excelente equilibrio entre **latencia** y **calidad de resultados**,
  - adecuado para catálogos pequeños y medianos.
- En este proyecto:
  - p50 ≈ **94.8 ms**,
  - p95 ≈ **103.5 ms**,
  - latencia estable y predecible.

**Por qué importa:**  
Permite búsquedas semánticas rápidas sin sacrificar demasiado la calidad del ranking.

---

#### 2. Integración de vectores + metadatos

- Qdrant permite almacenar:
  - el **vector** del producto,
  - y un **payload estructurado** con:
    - `record_id`,
    - `product_id`,
    - `title`,
    - `brand`,
    - `color`,
    - `active`,
    - `catalog_version`.
- Los filtros (`make_filter`) se aplican directamente sobre estos metadatos:
  - marca,
  - color,
  - estado activo,
  - versión de catálogo.

**Por qué importa:**  
Permite combinar búsqueda semántica con filtrado estructurado en una sola operación, sin necesidad de orquestar varias bases de datos.

---

#### 3. API sencilla y orientada a producción

- Qdrant ofrece:
  - API HTTP clara,
  - SDKs en Python,
  - operaciones de:
    - creación de colecciones,
    - upsert de puntos,
    - búsqueda,
    - filtrado,
    - borrado.
- En el proyecto:
  - funciones como `create_qdrant_collection`, `upsert_vector`, `search`, `apply_catalog_events` encapsulan estas operaciones.

**Por qué importa:**  
Facilita un pipeline reproducible y mantenible, alineado con un entorno real de producción.

---

#### 4. Estabilidad y consistencia del índice

- Qdrant mantiene:
  - consistencia del índice vectorial,
  - idempotencia en la aplicación de eventos:
    - los eventos ya aplicados se detectan y se ignoran.
- Esto se refleja en:
  - logs de eventos (`Evento X ya aplicado → ignorado`),
  - sincronización estable del catálogo.

**Por qué importa:**  
Garantiza que el estado del sistema sea coherente, incluso tras múltiples ejecuciones del pipeline.

---

#### 5. Adecuación al objetivo del proyecto

El objetivo del proyecto no es solo “almacenar vectores”, sino:

- implementar un **buscador semántico**,
- soportar **control de duplicados**,
- combinar **ranking vectorial** con **metadatos**,
- medir **latencia**, **fidelidad ANN** y **métricas de ranking**.

Qdrant encaja bien porque:

- está diseñado específicamente para **búsqueda vectorial**,
- soporta **payloads estructurados**,
- ofrece **filtros nativos**,
- permite **índices ANN** eficientes,
- se integra de forma natural con modelos de embeddings como E5-small.

---

### Resumen

- Es un **buscador semántico**:
  - consultas → vectores → top‑k ordenado,
  - filtros por marca, color, active.
- Implementa **control de altas**:
  - nuevas fichas → similitud → detección de duplicados.
- Qdrant se ha elegido como base de datos vectorial por:
  - soporte de HNSW,
  - integración de vectores + metadatos,
  - API sencilla,
  - estabilidad del índice,
  - adecuación al objetivo del proyecto.

---